# **Procesamiento del lenguaje natural**

**Tabajo parcial II**

**Tema:** Agente Conversacional para Admisión y Nivelación de la UG

Integrantes:


*   Caicho Almeida Shanney
*   Cedeño Pinto Scarlett



##**1. Instalacion e importacion de bibliotecas**

In [1]:
# Importacion de librerias
!pip install nltk
!pip install git+https://github.com
!pip install nltk scikit-learn pandas gradio --quiet

import re
import json
import random
import unicodedata
import nltk #stopwords en espanol y stemmer Snowball
import pandas as pd #lectura de JSON y visualizacion tabular
import gradio as gr #interfaz web del agente conversacional

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score,f1_score,classification_report)

# Descarga de stopwords en espanol (solo necesaria la primera vez)
nltk.download('stopwords', quiet=True)

# Semilla global de reproducibilidad
# Se fija en 19 para que la seleccion aleatoria de respuestas (random.choice)
# sea siempre la misma entre ejecuciones distintas del notebook
SEMILLA = 19
random.seed(SEMILLA)

  Cloning https://github.com to /tmp/pip-req-build-3vv_x4nj
  Running command git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-3vv_x4nj
  fatal: repository 'https://github.com/' not found
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-3vv_x4nj did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com /tmp/pip-req-build-3vv_x4nj did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


##**2. Recoleccion y modelado de datos (RF-01)**

In [2]:
# RF-01 | Configuracion: URLs de los JSON alojados en GitHub
# Al separar los datos del codigo, agregar o editar intenciones y secciones
# del corpus no requiere modificar la logica del agente
URL_INTENCIONES = (
    "https://raw.githubusercontent.com/Shaoss3/PLN_Agente_Conversacional"
    "/refs/heads/main/data/intenciones.json"
)
URL_CORPUS = (
    "https://raw.githubusercontent.com/Shaoss3/PLN_Agente_Conversacional"
    "/refs/heads/main/data/corpus_nivelacion.json"
)

# Umbral de confianza: similitud coseno minima para aceptar una respuesta
# Por debajo de este valor el agente activa el mecanismo de fallback
# Un umbral de 0.25 es adecuado para chatbots de dominio cerrado:
# suficientemente bajo para no rechazar variaciones validas,
# suficientemente alto para evitar respuestas incorrectas
UMBRAL = 0.25

# Umbral independiente para la busqueda en el corpus documental
# Es mas bajo porque el corpus contiene parrafos largos y el vocabulario
# se dispersa mas que en los utterances cortos de intenciones.json
UMBRAL_CORPUS = 0.10

In [3]:
def cargar_datos(url):
    """
    RF-01 | Carga el archivo intenciones.json desde GitHub.

    Extrae tres estructuras:
      - utterances : lista de frases preprocesadas para entrenar el TF-IDF
      - etiquetas  : lista paralela con el tag de cada utterance
      - respuestas : diccionario {tag: [lista de respuestas]} para devolver al usuario

    Args:
        url (str): URL raw del archivo JSON en GitHub.

    Returns:
        tuple: (utterances, etiquetas, respuestas)
    """
    # Lectura del JSON como DataFrame; el campo 'intenciones' es la lista de objetos
    datos = pd.read_json(url)

    utterances = []  # frases de usuario preprocesadas
    etiquetas  = []  # tag correspondiente a cada frase
    respuestas = {}  # diccionario tag -> lista de respuestas predefinidas

    for intencion in datos['intenciones']:
        tag = intencion['tag']
        # Guardar respuestas asociadas a esta intencion
        respuestas[tag] = intencion['respuestas']
        # Preprocesar y registrar cada utterance con su etiqueta
        for frase in intencion['utterances']:
            utterances.append(preprocesar(frase))
            etiquetas.append(tag)

    return utterances, etiquetas, respuestas


def cargar_corpus(url_corpus):
    """
    RF-01 | Carga corpus_nivelacion.json y prepara sus secciones para busqueda.

    A diferencia de intenciones.json (frases cortas), el corpus contiene
    parrafos extensos que se usan para enriquecer la respuesta final,
    no para clasificar la intencion.

    Args:
        url_corpus (str): URL raw del archivo JSON del corpus en GitHub.

    Returns:
        tuple: (tags_corpus, textos_originales, contenidos_preprocesados)
    """
    datos = pd.read_json(url_corpus)

    tags_corpus         = []  # tag de cada seccion del corpus
    textos_originales    = []  # texto completo tal como se muestra al usuario
    contenidos_limpios   = []  # version preprocesada para vectorizar

    for seccion in datos['secciones']:
        tags_corpus.append(seccion['tag'])
        textos_originales.append(seccion['contenido'])
        contenidos_limpios.append(preprocesar(seccion['contenido']))

    return tags_corpus, textos_originales, contenidos_limpios

##**3. Preprocesamiento de texto (RF-02)**

In [4]:
# Inicializacion del stemmer en espanol (algoritmo Snowball)
# El stemmer reduce cada palabra a su raiz morfologica
stemmer_es = SnowballStemmer('spanish')

# Conjunto de stopwords en espanol: palabras de poco valor semantico
# (articulos, preposiciones, pronombres) que se eliminan para reducir ruido
STOPWORDS_ES = set(stopwords.words('spanish'))


def preprocesar(texto):
    """
    RF-02 | Limpia y normaliza un texto para su vectorizacion con TF-IDF.

    Aplica: minusculas -> quitar tildes -> quitar puntuacion
            -> tokenizacion -> eliminar stopwords -> stemming

    Args:
        texto (str): Texto crudo de un utterance, seccion de corpus o consulta.

    Returns:
        str: Texto normalizado listo para el vectorizador.
    """
    # Paso 1: convertir a minusculas para unificar capitalizacion
    texto = texto.lower()

    # Paso 2: eliminar tildes y acentos mediante normalizacion Unicode NFKD
    # Esto evita que 'admision' y 'admision' (con tilde) sean palabras distintas
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ascii', 'ignore').decode('ascii')

    # Paso 3: quitar todo lo que no sea letra o espacio (puntuacion, numeros, urls)
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)

    # Paso 4: tokenizar dividiendo por espacios en blanco
    tokens = texto.split()

    # Paso 5: eliminar stopwords; solo conservar palabras con carga semantica
    tokens = [t for t in tokens if t not in STOPWORDS_ES]

    # Paso 6: stemming; reduce cada palabra a su raiz para capturar variantes
    # Ejemplo: 'inscripcion', 'inscribirme', 'inscrito' -> 'inscrib'
    tokens = [stemmer_es.stem(t) for t in tokens]

    return ' '.join(tokens)

##**4. Extraccion de entidades (RF-05)**

In [5]:
#Listado de carreras reconocidas por el agente
# Basado en las carreras descritas en corpus_nivelacion.json
CARRERAS = [
    "arquitectura", "ciencia de datos e inteligencia artificial",
    "ingenieria civil", "ingenieria industrial", "mecatronica",
    "sistemas de informacion", "software", "tecnologias de la informacion",
    "telematica", "medicina", "enfermeria", "odontologia", "psicologia",
    "derecho", "administracion de empresas", "contabilidad y auditoria",
    "economia", "finanzas", "bioquimica y farmacia"
]

# Terminos especificos del proceso que el agente puede identificar
TERMINOS_PROCESO = [
    "cupo aceptado", "curso de nivelacion", "registro nacional",
    "siug", "matricula", "postulacion", "evaluacion de conocimientos",
    "bloque de conocimiento", "admision", "nivelacion", "campus virtual"
]


def extraer_entidades(texto):
    """
    RF-05 | Detecta entidades relevantes en la consulta original del usuario.

    Busca mediante comparacion de cadenas (texto no preprocesado) para preservar
    informacion como nombres de carreras, terminos del proceso y numeros de cedula.

    Args:
        texto (str): Texto crudo ingresado por el usuario.

    Returns:
        dict: Diccionario con las entidades encontradas.
              Ejemplo: {'carrera': 'medicina', 'termino_proceso': 'matricula'}
    """
    texto_lower = texto.lower()
    entidades = {}

    # Detectar si el usuario menciona una carrera especifica
    for carrera in CARRERAS:
        if carrera in texto_lower:
            entidades['carrera'] = carrera
            break  # solo se registra la primera carrera encontrada

    # Detectar si el usuario menciona un termino especifico del proceso
    for termino in TERMINOS_PROCESO:
        if termino in texto_lower:
            entidades['termino_proceso'] = termino
            break

    # Detectar numeros de cedula: patron de 10 digitos consecutivos
    cedula = re.search(r'\b\d{10}\b', texto)
    if cedula:
        entidades['cedula'] = cedula.group()

    return entidades


# Verificacion de extraccion de entidades
print("Prueba de extraccion de entidades:")
print("-" * 50)
pruebas_entidades = [
    "quiero saber sobre la carrera de medicina",
    "como es el proceso de matricula en la ug",
    "mi cedula es 0912345678 cuando me inscribo"
]
for p in pruebas_entidades:
    print(f"  Texto     : {p}")
    print(f"  Entidades : {extraer_entidades(p)}")
    print()

Prueba de extraccion de entidades:
--------------------------------------------------
  Texto     : quiero saber sobre la carrera de medicina
  Entidades : {'carrera': 'medicina'}

  Texto     : como es el proceso de matricula en la ug
  Entidades : {'termino_proceso': 'matricula'}

  Texto     : mi cedula es 0912345678 cuando me inscribo
  Entidades : {'cedula': '0912345678'}



##**5. Representacion TF-IDF y entrenamiento del agente (RF-03)**

In [6]:
def entrenar(textos):
    """
    RF-03 | Ajusta un vectorizador TF-IDF sobre una coleccion de textos.

    Se reutiliza tanto para los utterances de intenciones.json como para
    las secciones de corpus_nivelacion.json.

    Configuracion:
      - ngram_range=(1,2): unigramas y bigramas para capturar frases como
        'registro nacional', 'cupo aceptado', 'cuenta siug'
      - sublinear_tf=True : aplica 1+log(tf) en lugar de tf para suavizar
        el efecto de palabras muy frecuentes dentro de un mismo texto

    Args:
        textos (list[str]): Textos ya preprocesados.

    Returns:
        tuple: (vectorizador ajustado, matriz TF-IDF dispersa)
    """
    vectorizador = TfidfVectorizer(
        ngram_range=(1, 2),   # captura unigramas y bigramas
        sublinear_tf=True     # suavizado logaritmico de frecuencias
    )
    # fit_transform: aprende el vocabulario y convierte cada texto en vector
    matriz = vectorizador.fit_transform(textos)
    return vectorizador, matriz

In [7]:
#Carga de intenciones y entrenamiento del modelo TF-IDF de clasificacion
utterances, etiquetas, respuestas = cargar_datos(URL_INTENCIONES)
vectorizador, matriz = entrenar(utterances)

print(f"Intenciones definidas : {len(set(etiquetas))}")
print(f"Utterances totales    : {matriz.shape[0]}")
print(f"Vocabulario (tokens)  : {matriz.shape[1]}")
print()

# Visualizacion de una muestra de la matriz TF-IDF
# Se muestran los primeros 8 utterances y las 12 palabras con mayor varianza
df_muestra = pd.DataFrame(
    matriz[:8].toarray(),
    columns=vectorizador.get_feature_names_out(),
    index=etiquetas[:8]
)

Intenciones definidas : 14
Utterances totales    : 112
Vocabulario (tokens)  : 269



In [8]:
#Carga del corpus documental y entrenamiento de su propio vectorizador
# Este vectorizador se usa unicamente para enriquecer respuestas (seccion 7),
# no para clasificar la intencion del usuario
tags_corpus, textos_corpus, contenidos_corpus = cargar_corpus(URL_CORPUS)
vectorizador_corpus, matriz_corpus = entrenar(contenidos_corpus)

print(f"Secciones del corpus disponibles: {len(tags_corpus)}")
print(f"Vocabulario del corpus          : {matriz_corpus.shape[1]}")
print("Tags disponibles:", ", ".join(tags_corpus))

Secciones del corpus disponibles: 10
Vocabulario del corpus          : 766
Tags disponibles: oferta_academica, informacion_carrera, canal_oficial_ug, campus_virtual_nivelacion, proceso_retiro_nivelacion, clases_nivelacion, aprobacion_nivelacion, matricula_nivelacion, proceso_admision, asignacion_cupo


## **6. Deteccion de intencion y respuesta de fallback (RF-04 + RF-06)**

In [9]:
def detectar_intencion(consulta, vectorizador, matriz, etiquetas):
    """
    RF-04 | Determina la intencion de la consulta usando similitud coseno sobre TF-IDF.

    Args:
        consulta (str)    : Texto crudo del usuario.
        vectorizador      : Vectorizador TF-IDF ya ajustado (intenciones).
        matriz            : Matriz TF-IDF de todos los utterances.
        etiquetas (list)  : Lista paralela con el tag de cada utterance.

    Returns:
        tuple: (tag detectado, valor de similitud)
    """
    # Preprocesar la consulta con la misma transformacion aplicada en entrenamiento
    consulta_limpia = preprocesar(consulta)

    # Vectorizar la consulta usando el vocabulario aprendido (transform, no fit_transform)
    vector_consulta = vectorizador.transform([consulta_limpia])

    # Calcular similitud coseno entre la consulta y todos los utterances
    # cosine_similarity devuelve un array de shape (1, n_utterances)
    similitudes = cosine_similarity(vector_consulta, matriz)[0]

    # Seleccionar el utterance con mayor similitud
    idx_mejor = similitudes.argmax()

    return etiquetas[idx_mejor], similitudes[idx_mejor]

In [10]:
def responder(consulta, vectorizador, matriz, etiquetas, respuestas,
              usar_corpus=True):
    """
    RF-04 + RF-05 + RF-06 + RF-07 | Genera la respuesta completa del agente.

    Flujo:
      1. Valida la entrada (RF-06).
      2. Detecta la intencion y calcula su similitud (RF-04).
      3. Extrae entidades del texto original (RF-05).
      4. Si la similitud supera UMBRAL, arma la respuesta base y la enriquece
         con el corpus documental cuando exista informacion relevante (RF-07).
      5. Si la similitud no supera UMBRAL, devuelve el mensaje de fallback.

    Args:
        consulta (str)      : Texto crudo del usuario.
        vectorizador, matriz, etiquetas, respuestas: modelo TF-IDF de intenciones.
        usar_corpus (bool)  : Si es False, omite la busqueda en el corpus
                              (util para evaluacion rapida de metricas).

    Returns:
        dict: {'tag', 'similitud', 'respuesta', 'entidades'}
    """
    # RF-06 | Manejo seguro de entradas vacias, None o texto problematico
    try:
        if not consulta or not consulta.strip():
            raise ValueError("Consulta vacia")

        tag, similitud = detectar_intencion(consulta, vectorizador, matriz, etiquetas)
        entidades = extraer_entidades(consulta)

        # RF-06 | Aplicar umbral de confianza
        if similitud >= UMBRAL:
            respuesta = random.choice(respuestas[tag])

            # RF-05 | Personalizar respuesta si se detecto una carrera
            if entidades.get('carrera'):
                respuesta += f"\n(Carrera detectada: {entidades['carrera'].title()})"

            # RF-07 | Enriquecer con el corpus documental si aplica
            if usar_corpus:
                fragmento = buscar_en_corpus(consulta, tag)
                if fragmento:
                    respuesta += f"\n\nInformacion adicional:\n{fragmento}"
        else:
            tag = 'fallback'
            respuesta = (
                "Lo siento, no entendi tu consulta. "
                "Puedes reformularla? Solo puedo responder preguntas "
                "sobre admision y nivelacion de la Universidad de Guayaquil."
            )

    # RF-06 | try-except: el agente nunca se detiene por una entrada problematica
    except Exception:
        tag = 'fallback'
        similitud = 0.0
        entidades = {}
        respuesta = (
            "Ocurrio un error al procesar tu consulta. "
            "Por favor intenta nuevamente con otra pregunta."
        )

    return {
        'tag'      : tag,
        'similitud': round(float(similitud), 4),
        'respuesta': respuesta,
        'entidades': entidades
    }

##**7. Enriquecimiento de respuestas con el corpus documental (RF-07)**

In [11]:
def buscar_en_corpus(consulta, tag_detectado):
    """
    RF-07 | Busca el fragmento mas relevante del corpus para una consulta.

    Prioriza la seccion cuyo tag coincide con la intencion ya detectada
    aplicando un factor de impulso (boost) sobre su similitud.

    Args:
        consulta (str)      : Texto crudo del usuario.
        tag_detectado (str) : Intencion detectada previamente por detectar_intencion().

    Returns:
        str | None: El parrafo mas relevante del corpus, o None si ninguna
                     seccion supera UMBRAL_CORPUS.
    """
    consulta_limpia = preprocesar(consulta)
    vector = vectorizador_corpus.transform([consulta_limpia])
    similitudes = cosine_similarity(vector, matriz_corpus)[0]

    # Impulso a la seccion del corpus que coincide con la intencion detectada
    for i, tag in enumerate(tags_corpus):
        if tag == tag_detectado:
            similitudes[i] *= 1.5
            break

    idx_mejor = similitudes.argmax()
    sim_mejor = similitudes[idx_mejor]

    if sim_mejor >= UMBRAL_CORPUS:
        return textos_corpus[idx_mejor]
    return None


# Verificacion rapida del enriquecimiento con corpus
print("Prueba de busqueda en el corpus documental:")
print("-" * 55)
for consulta_prueba in ["cuanto dura la carrera de medicina",
                        "como me retiro de nivelacion"]:
    tag_prueba, _ = detectar_intencion(consulta_prueba, vectorizador, matriz, etiquetas)
    fragmento = buscar_en_corpus(consulta_prueba, tag_prueba)
    print(f"  Consulta : {consulta_prueba}")
    print(f"  Tag      : {tag_prueba}")
    print(f"  Fragmento: {(fragmento or 'sin coincidencia')[:120]}...")
    print()

Prueba de busqueda en el corpus documental:
-------------------------------------------------------
  Consulta : cuanto dura la carrera de medicina
  Tag      : informacion_carrera
  Fragmento: La Universidad de Guayaquil cuenta con 17 facultades y 59 carreras vigentes con modalidades de estudio presencial, híbri...

  Consulta : como me retiro de nivelacion
  Tag      : matricula_nivelacion
  Fragmento: El retiro voluntario del curso de nivelación se puede solicitar dentro de las fechas establecidas de manera online a tra...



##**8. Prueba del agente (RF-08)**

In [12]:
# Pruebas rapidas de la funcion responder() de extremo a extremo
print("Pruebas de respuesta del agente:")
print("=" * 60)
pruebas = [
    "como me matriculo en nivelacion",
    "cuanto es 2 mas 2",          # fuera de dominio
    "",                            # entrada vacia
    "q bloque de conocimiento soy"
]
for p in pruebas:
    r = responder(p, vectorizador, matriz, etiquetas, respuestas)
    print(f"Consulta  : {repr(p)}")
    print(f"Intencion : {r['tag']} (sim={r['similitud']})")
    print(f"Respuesta : {r['respuesta'][:100]}...")
    print()

Pruebas de respuesta del agente:
Consulta  : 'como me matriculo en nivelacion'
Intencion : matricula_nivelacion (sim=1.0)
Respuesta : La matrícula al curso de nivelación es online a través del SIUG, según el periodo, carrera y jornada...

Consulta  : 'cuanto es 2 mas 2'
Intencion : fuera_dominio (sim=1.0)
Respuesta : Lo siento, solo puedo responder preguntas relacionadas con el proceso de admisión y nivelación de la...

Consulta  : ''
Intencion : fallback (sim=0.0)
Respuesta : Ocurrio un error al procesar tu consulta. Por favor intenta nuevamente con otra pregunta....

Consulta  : 'q bloque de conocimiento soy'
Intencion : aplicacion_evaluaciones (sim=1.0)
Respuesta : El examen de admisión evalúa conocimientos y habilidades según el perfil de ingreso de la carrera se...



In [13]:
#Conjunto de prueba: consultas variadas alineadas con los tags
# reales de intenciones.json
conjunto_prueba = [
    # --- Variaciones linguisticas normales ---
    ("como puedo registrarme en la ug",             "proceso_admision"),
    ("cuales son los pasos para entrar a la ug",    "proceso_admision"),
    ("q documentos necesito para inscribirme",      "requisitos_admision"),
    ("que documentos pido para entrar a la ug",     "requisitos_admision"),
    ("en q fecha me registro segun mi cedula",      "fechas_registro"),
    ("cuando son las fechas de postulacion",        "fechas_registro"),
    ("como se crea una cuenta en el siug",          "crear_cuenta_siug"),
    ("no tengo usuario en el siug q hago",          "crear_cuenta_siug"),
    ("que toman en el examen de ingreso",           "aplicacion_evaluaciones"),
    ("como es la prueba de ingreso",                "aplicacion_evaluaciones"),
    ("como me asignan el cupo",                     "asignacion_cupo"),
    ("con q puntaje me dan el cupo",                "asignacion_cupo"),
    ("como me matriculo en nivelacion",             "matricula_nivelacion"),
    ("donde hago la matricula del curso",           "matricula_nivelacion"),
    ("con cuanto se aprueba nivelacion",            "aprobacion_nivelacion"),
    ("que nota necesito para pasar nivelacion",     "aprobacion_nivelacion"),
    ("cuando son las clases",                       "clases_nivelacion"),
    ("como entro a zoom para mis clases",           "clases_nivelacion"),
    # --- Errores tipograficos intencionales ---
    ("cuando son las fechas de inscripsion",        "fechas_registro"),
    ("con cuanto apruebo nivelasion",               "aprobacion_nivelacion"),
    ("como se si aprobe nibelacion",                "aprobacion_nivelacion"),
    # --- Oferta academica e informacion de carrera ---
    ("que carreras ofrece la ug",                   "oferta_academica"),
    ("cuantas facultades tiene la universidad",     "oferta_academica"),
    ("cuanto dura la carrera de medicina",          "informacion_carrera"),
    ("que titulo obtengo en derecho",               "informacion_carrera"),
    # --- Saludos y despedidas ---
    ("buenas tardes",                               "saludo"),
    ("hola necesito ayuda con admision",            "saludo"),
    ("muchas gracias eso era todo",                 "despedida"),
    ("ok ya entendi chao",                          "despedida"),
    # --- Fuera de dominio ---
    ("recomiendame una pelicula",                   "fuera_dominio"),
    ("quien gano el mundial",                       "fuera_dominio"),
]

print(f"Total de consultas de prueba: {len(conjunto_prueba)}")

Total de consultas de prueba: 31


In [14]:
def evaluar_agente(conjunto_prueba, vectorizador, matriz, etiquetas, respuestas):
    """
    RF-08 | Evalua el agente sobre el conjunto de prueba y calcula metricas.

    Metricas calculadas:
      - Accuracy : porcentaje de intenciones correctamente detectadas.
      - F1-macro : promedio del F1 por clase; penaliza el desbalance entre intenciones.

    Args:
        conjunto_prueba (list[tuple]): Lista de (consulta, intencion_esperada).

    Returns:
        pd.DataFrame: Tabla con columnas Consulta, Esperada, Predicha, Correcta.
    """
    reales    = []  # intenciones esperadas segun el conjunto de prueba
    predichas = []  # intenciones que el agente predijo

    for consulta, intencion_real in conjunto_prueba:
        # usar_corpus=False: la evaluacion mide clasificacion, no enriquecimiento
        resultado = responder(consulta, vectorizador, matriz, etiquetas, respuestas,
                               usar_corpus=False)
        reales.append(intencion_real)
        predichas.append(resultado['tag'])

    # Calculo de metricas globales
    acc = accuracy_score(reales, predichas)
    f1  = f1_score(reales, predichas, average='macro', zero_division=0)

    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.2%}")
    print(f"  F1-macro  : {f1:.2%}")
    print(f"{'='*50}")
    print()
    print("Reporte detallado por intencion:")
    print(classification_report(reales, predichas, zero_division=0))

    # Tabla de resultados individuales
    df = pd.DataFrame({
        'Consulta': [c for c, _ in conjunto_prueba],
        'Esperada': reales,
        'Predicha': predichas,
        'Correcta': ['SI' if r == p else 'NO' for r, p in zip(reales, predichas)]
    })
    return df


df_resultados = evaluar_agente(
    conjunto_prueba, vectorizador, matriz, etiquetas, respuestas
)

  Accuracy  : 90.32%
  F1-macro  : 86.19%

Reporte detallado por intencion:
                         precision    recall  f1-score   support

aplicacion_evaluaciones       1.00      1.00      1.00         2
  aprobacion_nivelacion       1.00      1.00      1.00         4
        asignacion_cupo       1.00      1.00      1.00         2
      clases_nivelacion       1.00      1.00      1.00         2
      crear_cuenta_siug       1.00      0.50      0.67         2
              despedida       1.00      1.00      1.00         2
        fechas_registro       1.00      1.00      1.00         3
          fuera_dominio       0.67      1.00      0.80         2
    informacion_carrera       1.00      1.00      1.00         2
   matricula_nivelacion       1.00      1.00      1.00         2
       oferta_academica       0.67      1.00      0.80         2
       proceso_admision       0.00      0.00      0.00         2
    requisitos_admision       0.67      1.00      0.80         2
             

In [15]:
# Visualizacion de la tabla completa de resultados
print("Tabla de resultados:")
print(df_resultados.to_string(index=False))
print()

# Mostrar solo los casos incorrectos si los hay
incorrectas = df_resultados[df_resultados['Correcta'] == 'NO']
if len(incorrectas) == 0:
    print("Todas las consultas de prueba fueron clasificadas correctamente.")
else:
    print(f"Predicciones incorrectas ({len(incorrectas)}):")
    print(incorrectas[['Consulta', 'Esperada', 'Predicha']].to_string(index=False))

Tabla de resultados:
                                Consulta                Esperada                Predicha Correcta
         como puedo registrarme en la ug        proceso_admision        oferta_academica       NO
cuales son los pasos para entrar a la ug        proceso_admision     requisitos_admision       NO
  q documentos necesito para inscribirme     requisitos_admision     requisitos_admision       SI
 que documentos pido para entrar a la ug     requisitos_admision     requisitos_admision       SI
  en q fecha me registro segun mi cedula         fechas_registro         fechas_registro       SI
    cuando son las fechas de postulacion         fechas_registro         fechas_registro       SI
      como se crea una cuenta en el siug       crear_cuenta_siug       crear_cuenta_siug       SI
      no tengo usuario en el siug q hago       crear_cuenta_siug           fuera_dominio       NO
       que toman en el examen de ingreso aplicacion_evaluaciones aplicacion_evaluaciones       SI

##**9. Interfaz web con Gradio (RF-09)**

In [ ]:
# RF-09 | Funcion de chat para la interfaz Gradio
def chat_gradio(mensaje, historial):
    """
    Procesa el mensaje del usuario y retorna la respuesta del agente.
    Compatible con el formato de ChatInterface de Gradio.

    Args:
        mensaje (str)   : Texto ingresado por el usuario.
        historial (list): Historial de la conversacion (gestionado por Gradio).

    Returns:
        str: Respuesta del agente lista para mostrar en la interfaz.
    """
    resultado = responder(mensaje, vectorizador, matriz, etiquetas, respuestas,
                           usar_corpus=True)

    # Mostrar intencion y similitud como informacion adicional en la respuesta
    info = (
        f"\n\n---\n"
        f"*Intencion detectada: `{resultado['tag']}` | "
        f"Similitud: `{resultado['similitud']}`*"
    )
    return resultado['respuesta'] + info


# Configuracion de la interfaz Gradio ChatInterface
# Nota: se quito el argumento 'theme' porque la version de Gradio instalada
# en este entorno no lo acepta en ChatInterface y provocaba TypeError
interfaz = gr.ChatInterface(
    fn=chat_gradio,
    title="Asistente de Admision y Nivelacion - Universidad de Guayaquil",
    description=(
        "Consulta informacion oficial sobre admision y nivelacion de la UG. "
        "Puedes preguntar sobre requisitos, fechas, cupos, matricula, clases, "
        "oferta academica y mas."
    ),
    examples=[
        "Como me matriculo en nivelacion?",
        "Con cuanto se aprueba nivelacion?",
        "Cuales son los requisitos para inscribirme?",
        "Cuanto dura la carrera de medicina?",
        "Que carreras ofrece la ug?",
    ]
)

# Lanzar la interfaz (share=False para ejecucion local; share=True para URL publica)
interfaz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://397b054e9644599c0e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
